# Forest Cover Type Classification: Model Comparison

This notebook adapts the analysis workflow from the Zomato notebook to the UCI Forest Covertype dataset. The target is **Cover_Type** (seven forest classes), so this is a multiclass classification problem. The notebook compares three models with different strengths: multinomial logistic regression, random forest, and histogram gradient boosting.

The dataset has 581,012 observations, 10 continuous terrain measurements, 4 wilderness indicators, and 40 soil indicators. The supplied ZIP contains `covtype.data.gz`; keep `covertype.zip` beside this notebook or set `DATA_ZIP` below to its path. No network download is required.


## 1. Imports and reproducibility

In [ ]:
# If needed, uncomment and run once:
# %pip install pandas numpy matplotlib seaborn scikit-learn

from pathlib import Path
from zipfile import ZipFile
import gzip
import io
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_score,
    recall_score, f1_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay, roc_auc_score, roc_curve, auc, make_scorer)
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42
TEST_SIZE = 0.20
DATA_ZIP = Path("covertype.zip")  # Update this path if the ZIP is elsewhere.
pd.set_option("display.max_columns", 70)
sns.set_theme(style="whitegrid")


## 2. Load the local dataset

Keep `covertype.zip` beside the notebook or set `DATA_ZIP` below. In Google Colab, running the loading cell opens an upload picker if the ZIP is not already present. The notebook reads `covtype.data.gz` directly from the ZIP; no separate extraction is needed.

In [ ]:
if not DATA_ZIP.exists():
    # Also check the common Downloads location for local notebook runs.
    downloads_zip = Path.home() / "Downloads" / "covertype.zip"
    if downloads_zip.exists():
        DATA_ZIP = downloads_zip

if not DATA_ZIP.exists():
    # In Google Colab, open the upload picker if the ZIP is not already in /content.
    try:
        from google.colab import files
        print("Select the covertype.zip file to upload it into this runtime.")
        uploaded_files = files.upload()
        zip_files = [name for name in uploaded_files if name.lower().endswith(".zip")]
        if not zip_files:
            raise FileNotFoundError("No ZIP file was uploaded. Upload covertype.zip and rerun this cell.")
        DATA_ZIP = Path(zip_files[0])
    except ImportError as exc:
        raise FileNotFoundError(
            f"Cannot find {DATA_ZIP.resolve()}. Put covertype.zip beside this notebook "
            "or set DATA_ZIP to the ZIP file path."
        ) from exc

column_names = [
    "Elevation", "Aspect", "Slope", "Horizontal_Distance_To_Hydrology",
    "Vertical_Distance_To_Hydrology", "Horizontal_Distance_To_Roadways",
    "Hillshade_9am", "Hillshade_Noon", "Hillshade_3pm",
    "Horizontal_Distance_To_Fire_Points",
    *[f"Wilderness_Area_{i}" for i in range(1, 5)],
    *[f"Soil_Type_{i}" for i in range(1, 41)],
    "Cover_Type",
]

with ZipFile(DATA_ZIP) as archive:
    with archive.open("covtype.data.gz") as compressed:
        with gzip.GzipFile(fileobj=compressed) as source:
            df = pd.read_csv(source, header=None, names=column_names)

print(f"Loaded {df.shape[0]:,} rows and {df.shape[1]} columns from {DATA_ZIP.name}")
df.head()

## 3. Initial inspection and data quality

In [ ]:
print("Shape:", df.shape)
print("\nData types:\n", df.dtypes.value_counts())
display(df.head())
display(df.describe().T)
print("Missing values:", int(df.isna().sum().sum()))
print("Duplicate rows:", int(df.duplicated().sum()))
print("Target values:", sorted(df["Cover_Type"].unique()))

The source documentation reports no missing values. We still check programmatically. Exact duplicate feature-target rows are reported, but retained: identical measurements can represent separate map cells, and removing them could change the observed class frequencies. The target is already categorical (classes 1Ã¢â‚¬â€œ7), so no artificial binary threshold is needed.

## 4. Target distribution and feature overview

In [ ]:
cover_names = {
    1: "Spruce/Fir", 2: "Lodgepole Pine", 3: "Ponderosa Pine",
    4: "Cottonwood/Willow", 5: "Aspen", 6: "Douglas-fir", 7: "Krummholz"
}
class_counts = df["Cover_Type"].value_counts().sort_index()
display(pd.DataFrame({"Cover type": [cover_names[i] for i in class_counts.index],
                      "Count": class_counts.values,
                      "Percent": (class_counts.values / len(df) * 100).round(2)},
                     index=class_counts.index).rename_axis("Class"))

ax = class_counts.rename(index=cover_names).plot(kind="bar", figsize=(10, 4), color="steelblue")
ax.set(title="Forest cover type distribution", xlabel="Cover type", ylabel="Observations")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
continuous_features = [
    "Elevation", "Aspect", "Slope", "Horizontal_Distance_To_Hydrology",
    "Vertical_Distance_To_Hydrology", "Horizontal_Distance_To_Roadways",
    "Hillshade_9am", "Hillshade_Noon", "Hillshade_3pm",
    "Horizontal_Distance_To_Fire_Points"
]
fig, axes = plt.subplots(2, 5, figsize=(16, 7))
for feature, ax in zip(continuous_features, axes.flat):
    sns.histplot(data=df, x=feature, bins=35, ax=ax, color="teal")
    ax.set_title(feature.replace("_", " "), fontsize=9)
plt.tight_layout()
plt.show()


In [ ]:
# Soil/wilderness columns are one-hot indicators. Check that each row has one active
# wilderness area and one active soil category.
wilderness_cols = [c for c in df.columns if c.startswith("Wilderness_Area_")]
soil_cols = [c for c in df.columns if c.startswith("Soil_Type_")]
print("Wilderness indicators per row:\n", df[wilderness_cols].sum(axis=1).value_counts().sort_index())
print("Soil indicators per row (most common):\n", df[soil_cols].sum(axis=1).value_counts().sort_index().head())

## 5. Define features, target, and holdout split

In [ ]:
X = df.drop(columns="Cover_Type")
y = df["Cover_Type"].astype(int)
class_labels = sorted(y.unique())
class_names = [cover_names[c] for c in class_labels]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)
print("Training rows:", f"{len(X_train):,}")
print("Test rows:", f"{len(X_test):,}")
print("Features:", X.shape[1], "| Classes:", len(class_labels))


The stratified 80/20 split preserves the class proportions. All preprocessing and model selection are fitted using training data only; the holdout set is reserved for the final comparison. These features are already numeric, and the 44 wilderness/soil indicators are already one-hot encoded.

## 6. Preprocessing pipelines

In [ ]:
# Imputation is included defensively. Scaling is useful for the linear model;
# tree models do not require scaling.
linear_preprocessor = ColumnTransformer([
    ("numeric", Pipeline([("imputer", SimpleImputer(strategy="median")),
                           ("scaler", StandardScaler())]), continuous_features),
    ("indicators", SimpleImputer(strategy="most_frequent"), wilderness_cols + soil_cols),
], remainder="drop")

tree_preprocessor = ColumnTransformer([
    ("numeric", SimpleImputer(strategy="median"), continuous_features),
    ("indicators", SimpleImputer(strategy="most_frequent"), wilderness_cols + soil_cols),
], remainder="drop")


## 7. Three classification models

1. **Multinomial Logistic Regression** provides a regularized linear baseline and benefits from standardized measurements.
2. **Random Forest** captures nonlinear relationships and feature interactions with bagged decision trees.
3. **Histogram Gradient Boosting** builds trees sequentially and is a strong fit for a large tabular dataset.

The target is imbalanced, so the comparison reports macro as well as weighted metrics. `class_weight='balanced'` is enabled for the first two classifiers; histogram gradient boosting has no class-weight argument in all supported scikit-learn versions, so its per-class and macro scores remain visible in evaluation.

In [ ]:
models = {
    "Multinomial Logistic Regression": Pipeline([
        ("preprocessor", linear_preprocessor),
        ("classifier", LogisticRegression(
            solver="lbfgs", max_iter=1000, class_weight="balanced",
            random_state=RANDOM_STATE, n_jobs=-1
        )),
    ]),
    "Random Forest": Pipeline([
        ("preprocessor", tree_preprocessor),
        ("classifier", RandomForestClassifier(
            n_estimators=250, min_samples_leaf=2, class_weight="balanced_subsample",
            random_state=RANDOM_STATE, n_jobs=-1
        )),
    ]),
    "Histogram Gradient Boosting": Pipeline([
        ("preprocessor", tree_preprocessor),
        ("classifier", HistGradientBoostingClassifier(
            max_iter=150, learning_rate=0.1, max_leaf_nodes=31,
            l2_regularization=1.0, random_state=RANDOM_STATE
        )),
    ]),
}


## 8. Cross-validation on the training partition

In [ ]:
# Five-fold stratified CV offers a training-only estimate. This can take several
# minutes because the dataset is large and three models are evaluated.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
    "macro_f1": "f1_macro",
    "weighted_f1": "f1_weighted",
}
cv_rows = []
for name, model in models.items():
    scores = cross_validate(model, X_train, y_train, cv=cv, scoring=scoring,
                            n_jobs=1, return_train_score=False)
    row = {"Model": name}
    for metric in scoring:
        vals = scores[f"test_{metric}"]
        row[f"CV {metric} mean"] = vals.mean()
        row[f"CV {metric} std"] = vals.std()
    cv_rows.append(row)
cv_results = pd.DataFrame(cv_rows).sort_values("CV macro_f1 mean", ascending=False)
display(cv_results.round(4))


## 9. Fit models and evaluate on the holdout set

In [ ]:
fitted_models = {}
predictions = {}
probabilities = {}
result_rows = []

for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train, y_train)
    fitted_models[name] = model
    pred = model.predict(X_test)
    predictions[name] = pred
    probabilities[name] = model.predict_proba(X_test)
    result_rows.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Balanced Accuracy": balanced_accuracy_score(y_test, pred),
        "Macro Precision": precision_score(y_test, pred, average="macro", zero_division=0),
        "Macro Recall": recall_score(y_test, pred, average="macro", zero_division=0),
        "Macro F1": f1_score(y_test, pred, average="macro", zero_division=0),
        "Weighted F1": f1_score(y_test, pred, average="weighted", zero_division=0),
        "ROC-AUC OVR macro": roc_auc_score(y_test, probabilities[name],
                                           multi_class="ovr", average="macro"),
    })

results = pd.DataFrame(result_rows).sort_values("Macro F1", ascending=False).reset_index(drop=True)
display(results.round(4))


## 10. Classification reports and confusion matrices

In [ ]:
for name, pred in predictions.items():
    print(f"\n{name} Ã¢â‚¬â€ per-class report")
    print(classification_report(y_test, pred, labels=class_labels,
                                target_names=class_names, zero_division=0, digits=3))


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(19, 5))
for ax, (name, pred) in zip(axes, predictions.items()):
    cm = confusion_matrix(y_test, pred, labels=class_labels, normalize="true")
    disp = ConfusionMatrixDisplay(cm, display_labels=class_names)
    disp.plot(ax=ax, cmap="Blues", colorbar=False, values_format=".2f", xticks_rotation=75)
    ax.set_title(name)
    ax.set_xlabel("Predicted class")
    ax.set_ylabel("True class")
plt.tight_layout()
plt.show()


## 11. Compare model metrics

In [ ]:
metric_columns = ["Accuracy", "Balanced Accuracy", "Macro F1", "Weighted F1", "ROC-AUC OVR macro"]
ax = results.set_index("Model")[metric_columns].plot(kind="bar", figsize=(12, 6))
ax.set(title="Holdout performance comparison", ylabel="Score", ylim=(0, 1))
plt.xticks(rotation=20, ha="right")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()


## 12. Multiclass ROC curves

In [ ]:
# One-vs-rest ROC curves show performance separately for each cover type.
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for class_id, ax in zip(class_labels, axes.flat):
    binary_y = (y_test == class_id).astype(int)
    for name in models:
        class_index = class_labels.index(class_id)
        fpr, tpr, _ = roc_curve(binary_y, probabilities[name][:, class_index])
        ax.plot(fpr, tpr, label=name)
    ax.plot([0, 1], [0, 1], "k--", linewidth=0.8)
    ax.set_title(f"{class_id}: {cover_names[class_id]}")
    ax.set_xlabel("False positive rate")
    ax.set_ylabel("True positive rate")
axes.flat[-1].axis("off")
handles, labels = axes.flat[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower right", bbox_to_anchor=(0.99, 0.12))
plt.tight_layout(rect=(0, 0, 0.88, 1))
plt.show()


## 13. Feature interpretation

In [ ]:
# Random Forest impurity importance is quick to compute. Treat it as descriptive:
# correlated terrain variables can divide or distort importance.
rf = fitted_models["Random Forest"].named_steps["classifier"]
rf_importance = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
display(rf_importance.head(20).rename("Importance").to_frame())
rf_importance.head(20).sort_values().plot(kind="barh", figsize=(9, 7), color="darkorange")
plt.title("Random Forest: top 20 impurity importances")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()


In [ ]:
# Permutation importance measures holdout performance change after shuffling a feature.
# Run on a stratified sample to keep this interpretation step practical.
sample_n = min(10000, len(X_test))
X_interpret = X_test.sample(sample_n, random_state=RANDOM_STATE)
y_interpret = y_test.loc[X_interpret.index]
best_name = results.loc[0, "Model"]
perm = permutation_importance(
    fitted_models[best_name], X_interpret, y_interpret,
    scoring="f1_macro", n_repeats=3, random_state=RANDOM_STATE, n_jobs=-1
)
perm_importance = pd.Series(perm.importances_mean, index=X.columns).sort_values(ascending=False)
display(perm_importance.head(20).rename("Mean macro-F1 decrease").to_frame())
perm_importance.head(20).sort_values().plot(kind="barh", figsize=(9, 7), color="seagreen")
plt.title(f"Permutation importance on holdout sample Ã¢â‚¬â€ {best_name}")
plt.xlabel("Mean decrease in macro F1")
plt.tight_layout()
plt.show()


## 14. Select and summarize the best model

In [ ]:
best_row = results.loc[results["Macro F1"].idxmax()]
print("Selected by holdout Macro F1:", best_row["Model"])
print(best_row.to_string())
print("\nInterpretation: macro F1 gives each cover type equal weight and is useful here because\n"
      "the class counts differ substantially. Also inspect per-class recall and the confusion\n"
      "matrix before choosing a model for a real application.")

## Notes and limitations

- Results are estimates for this randomized stratified split of a spatial dataset. Nearby cells may be related; a spatially grouped evaluation would be a useful follow-up if the goal is prediction in unseen geographic areas.
- The seven cover types are imbalanced. Accuracy alone can hide weak performance on the smallest classes, so macro F1, balanced accuracy, and per-class recall are included.
- Model parameters are reasonable starting points, not exhaustively tuned. The next step for a production model would be a training-only hyperparameter search and a spatially aware validation design.
- The accompanying archive documentation describes the source variables and class meanings. This notebook uses the class mapping provided with the archive.